# NYC Yellow Taxi — Operational Dashboard

**Tickets:** A-05, A-06, D-01  
**Data range:** January 2015 + January–March 2016 (composite)  
**Source:** Gold fact table (`gold_fact_trips`) — aggregated from \~140M cleaned Silver trip records.

Use the **Day filter** widget above to toggle between All / Weekday / Weekend views.

---

## Setup

In [0]:
import importlib

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import src.constants

importlib.reload(src.constants)

from pyspark.sql import functions as F  # noqa: E402
from src.constants import DAY_NAME_MAP, GOLD_FACT_TABLE, TIME_PERIOD_BINS  # noqa: E402

# ── D-01: Unified colour palette ────────────────────────────────────────────
PALETTE = {
    "primary": "#1B4F72",  # deep navy  — revenue charts
    "secondary": "#E67E22",  # warm amber — duration charts
    "accent": "#27AE60",  # green      — trip count charts
    "highlight": "#C0392B",  # red        — callouts
    "heatmap": "YlOrRd",  # sequential — heatmap
    "bg": "#FAFAFA",  # light grey — figure background
}

# ── D-01: Consistent matplotlib theme ────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.facecolor": PALETTE["bg"],
        "axes.facecolor": "#FFFFFF",
        "axes.titlesize": 14,
        "axes.titleweight": "bold",
        "axes.labelsize": 11,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "grid.linestyle": "--",
        "legend.fontsize": 10,
        "figure.dpi": 110,
    }
)

# ── D-01: Interactive day filter ─────────────────────────────────────────────
# Creates widget on first run; preserves user-selected value on re-runs.
dbutils.widgets.dropdown(
    "day_filter", "All", ["All", "Weekday", "Weekend"], "Day filter"
)

day_filter = dbutils.widgets.get("day_filter")

# Read persisted Gold fact table & apply filter
fact_df_all = spark.read.table(GOLD_FACT_TABLE)

if day_filter == "Weekday":
    fact_df = fact_df_all.filter(F.col("is_weekend") == False)  # noqa: E712
elif day_filter == "Weekend":
    fact_df = fact_df_all.filter(F.col("is_weekend") == True)  # noqa: E712
else:
    fact_df = fact_df_all

row_count = fact_df.count()
filter_label = f"{day_filter} days" if day_filter != "All" else "All days"

print("\u2713 Setup complete")
print(f"  Gold fact table: {GOLD_FACT_TABLE}  ({row_count:,} rows)")
print(f"  Filter: {filter_label}")

## A-05 — Demand heatmap (zone × hour-of-day)

In [0]:
# D-01 — Demand heatmap: top 15 zones × hour-of-day

# Identify the 15 highest-demand zones (within current filter)
top_zones = (
    fact_df.groupBy("pickup_zone")
    .agg(F.sum("trip_count").alias("total_trips"))
    .orderBy(F.col("total_trips").desc())
    .limit(15)
    .select("pickup_zone")
)

# Pivot: zone (rows) × hour (columns) → trip_count
heatmap_df = (
    fact_df.join(top_zones, on="pickup_zone")
    .groupBy("pickup_zone")
    .pivot("hour_of_day", list(range(24)))
    .agg(F.sum("trip_count"))
    .fillna(0)
    .toPandas()
)


def _clean_zone(label: str) -> str:
    parts = label.split(",")
    if len(parts) == 2:
        return f"{float(parts[0]):.2f},{float(parts[1]):.2f}"
    return label


heatmap_df["pickup_zone"] = heatmap_df["pickup_zone"].apply(_clean_zone)
heatmap_df = heatmap_df.set_index("pickup_zone")
heatmap_df = heatmap_df.loc[heatmap_df.sum(axis=1).sort_values(ascending=False).index]
heatmap_df.columns = [f"{int(c):02d}:00" for c in heatmap_df.columns]

# Plot
fig, ax = plt.subplots(figsize=(18, 7))
sns.heatmap(
    heatmap_df,
    cmap=PALETTE["heatmap"],
    fmt=",.0f",
    annot=False,
    linewidths=0.3,
    ax=ax,
    cbar_kws={"label": "Trip count"},
)
ax.set_title(
    f"Taxi Demand Heatmap: Top 15 Zones \u00d7 Hour of Day\n({filter_label})",
    fontsize=15,
    weight="bold",
    pad=12,
)
ax.set_xlabel("Hour of Day", fontsize=11)
ax.set_ylabel("Pickup Zone (lat, lon)", fontsize=11)
plt.tight_layout()
plt.show()

## A-05 — KPI cards

In [0]:
# D-01 — KPI cards + polished charts
import pandas as pd

# ── Headline KPIs (filter-aware) ─────────────────────────────────────────────
kpis = fact_df.agg(
    F.sum("trip_count").alias("total_trips"),
    F.sum("total_revenue").alias("total_revenue"),
    F.round(F.sum("total_revenue") / F.sum("trip_count"), 2).alias(
        "avg_revenue_per_trip"
    ),
    F.round(
        F.sum(F.col("avg_trip_duration_min") * F.col("trip_count"))
        / F.sum("trip_count"),
        2,
    ).alias("weighted_avg_duration_min"),
).first()

print("\u2550" * 60)
print(f"  HEADLINE KPIs — {filter_label.upper()}")
print("\u2550" * 60)
print(f"  \U0001f696  Total trips      : {kpis['total_trips']:>14,}")
print(f"  \U0001f4b0  Total revenue    : ${kpis['total_revenue']:>14,.0f}")
print(f"  \U0001f4b5  Avg fare / trip  : ${kpis['avg_revenue_per_trip']:>10,.2f}")
print(f"  \u23f1   Avg duration     : {kpis['weighted_avg_duration_min']:>10.1f} min")
print("\u2550" * 60)

# ── Revenue by hour of day ─────────────────────────────────────────────────
rev_hour_pd = (
    fact_df.groupBy("hour_of_day")
    .agg(F.sum("total_revenue").alias("total_revenue"))
    .orderBy("hour_of_day")
    .toPandas()
)

# ── Avg trip duration by day of week ──────────────────────────────────────
dur_day_pd = (
    fact_df.groupBy("day_of_week")
    .agg(
        F.round(
            F.sum(F.col("avg_trip_duration_min") * F.col("trip_count"))
            / F.sum("trip_count"),
            2,
        ).alias("avg_duration_min"),
    )
    .orderBy("day_of_week")
    .toPandas()
)
dur_day_pd["day_name"] = dur_day_pd["day_of_week"].map(DAY_NAME_MAP)

# ── Trips by time period (D-01 new panel) ───────────────────────────────
period_rows = []
for period_name, (start_h, end_h) in TIME_PERIOD_BINS.items():
    hours_in_period = list(range(start_h, end_h + 1))
    period_trips = (
        (
            fact_df.filter(F.col("hour_of_day").isin(hours_in_period))
            .agg(F.sum("trip_count").alias("trips"))
            .first()["trips"]
        )
        or 0
    )
    period_rows.append({"period": period_name, "trips": period_trips})

period_pd = pd.DataFrame(period_rows)
# Order: Morning, Afternoon, Evening, Night
period_order = ["Morning", "Afternoon", "Evening", "Night"]
period_pd["period"] = pd.Categorical(
    period_pd["period"], categories=period_order, ordered=True
)
period_pd = period_pd.sort_values("period")

# ── 3-panel chart ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))
fig.suptitle(
    f"NYC Taxi KPI Summary — {filter_label}", fontsize=16, weight="bold", y=1.02
)

# Panel 1: Revenue by hour
axes[0].bar(
    rev_hour_pd["hour_of_day"],
    rev_hour_pd["total_revenue"] / 1e6,
    color=PALETTE["primary"],
    edgecolor="white",
    linewidth=0.4,
)
axes[0].set_title("Revenue by Hour of Day")
axes[0].set_xlabel("Hour")
axes[0].set_ylabel("Revenue ($M)")
axes[0].set_xticks(range(0, 24, 2))
axes[0].yaxis.set_major_formatter(mticker.FormatStrFormatter("$%.0fM"))

# Panel 2: Avg duration by day
bar_colors = [
    PALETTE["highlight"] if d in (1, 7) else PALETTE["secondary"]
    for d in dur_day_pd["day_of_week"]
]
axes[1].bar(
    dur_day_pd["day_name"],
    dur_day_pd["avg_duration_min"],
    color=bar_colors,
    edgecolor="white",
    linewidth=0.4,
)
axes[1].set_title("Avg Trip Duration by Day")
axes[1].set_xlabel("Day")
axes[1].set_ylabel("Duration (min)")
axes[1].tick_params(axis="x", rotation=30)

# Panel 3: Trips by time period (new in D-01)
axes[2].bar(
    period_pd["period"],
    period_pd["trips"] / 1e6,
    color=PALETTE["accent"],
    edgecolor="white",
    linewidth=0.4,
)
axes[2].set_title("Trips by Time of Day")
axes[2].set_xlabel("Period")
axes[2].set_ylabel("Trips (millions)")
axes[2].yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1fM"))

plt.tight_layout()
plt.show()

## A-06 — Contextual narrative

**Why this matters:**

**Demand heatmap** — The heatmap reveals which lat/lon grid zones (\~1.1 km cells) generate the most taxi pickups and at which hours. Midtown Manhattan (`40.76, -73.97`) dominates across all hours, with a pronounced evening peak (18:00–21:00 on weekdays). Fleet operators can use this to pre-position vehicles in high-demand corridors before rush hour, reducing idle time and passenger wait.

**Revenue by hour** — Revenue peaks in the early evening (18:00–20:00) and drops sharply after midnight, tracking demand closely. A secondary morning peak (07:00–09:00) reflects the commute. Revenue per trip is relatively stable, meaning the revenue curve is driven primarily by volume, not price variation.

**Duration by day** — Average trip duration is broadly consistent across the week, with a slight uptick on weekends. This suggests congestion (which lengthens trips) is offset by lower weekend traffic. Drivers and dispatchers should expect similar per-trip time commitments regardless of the day.

### Data caveats

| Caveat | Detail |
|--------|--------|
| **Non-contiguous date range** | All metrics are a composite of January 2015 and January–March 2016 (9-month gap). Seasonal and year-over-year trends cannot be inferred. |
| **Grid-binned zones** | Zones are \~0.01\u00b0 lat/lon bins (\~1.1 km), not official TLC taxi zones or neighbourhood names. |
| **Cash tip blindspot** | 34% of trips paid by cash report `tip_amount = 0` (tips not recorded, not absent). Any tip-related metrics are systematically deflated. |